# Comparación de la heurística KNN con la sintonización individual y con un controlador de parámetros fijos

En esta libreta se evalúa la heurística de estimación de hiperparámetros frente a dos referencias, simulando las tres estrategias con la misma planta virtual, la  trayectoria de referencia Λ(t) y la misma implementación:

- **AG:** parámetros sintonizados individualmente mediante algoritmo genético (`SINTONIZACIONES.csv`).
- **KNN:** parámetros estimados por la heurística en su variante global (p = 1.4, k = 2).
- **Parámetros medianos:** un único controlador cuyos parámetros son la mediana, componente a componente, de los parámetros del espacio de referencia.

La comparación se efectúa sobre el conjunto de estimación (174 pacientes) y sobre el conjunto de casos con sintonización deficiente (RMSE del AG mayor que 16).



## Etapa 1: Configuración del entorno (tomada de `KNN_Estimacion_PID.ipynb`)

In [1]:
# =============================================================================
# CONFIGURACIÓN GLOBAL 
# =============================================================================

# ── Rutas ─────────────────────────────────────────────────────────────────────
RUTA_MODELO          = r"C:\Users\yordi\Desktop\modeloBIS.pt"
RUTA_GRAFICAS        = r"C:\Users\yordi\Desktop"
RUTA_SINTONIZACIONES = "SINTONIZACIONES.csv"   # Archivo generado en la libreta previa

# ── Rangos de normalización del modelo predictor del BIS ──────────────────────
# (fijos, correspondientes al preprocesamiento con el que fue entrenado)
EDAD_MIN, EDAD_MAX     = 16.00, 90.00
PESO_MIN, PESO_MAX     = 34.10, 112.80
ALTURA_MIN, ALTURA_MAX = 140.20, 186.60

# =============================================================================
# IMPORTACIONES
# =============================================================================
import os, time, math, sys
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch
import torch.nn as nn
from scipy import stats
from Modelo_Predictor_BIS import ModeloAnestesia

# =============================================================================
# DISPOSITIVO Y CARGA DEL MODELO PREDICTOR DEL BIS
# =============================================================================
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Dispositivo : {dispositivo}")
if dispositivo.type == "cuda":
    print(f"[INFO] GPU         : {torch.cuda.get_device_name(0)}")

modelo = ModeloAnestesia().to(dispositivo)
modelo.load_state_dict(
    torch.load(RUTA_MODELO, map_location=dispositivo, weights_only=True)
)
modelo.eval()
for p in modelo.parameters():
    p.requires_grad_(False)

print(f"[INFO] Modelo cargado — "
      f"{sum(p.numel() for p in modelo.parameters()):,} parámetros")


# =============================================================================
# NORMALIZACIÓN DE LAS COVARIABLES PARA EL MODELO PREDICTOR DEL BIS
# =============================================================================
# NOTA: esta normalización corresponde al preprocesamiento fijo del modelo
# predictor del BIS y es independiente de la normalización min-max que se
# aplicará posteriormente para el algoritmo de K vecinos más cercanos.

def normalizar_paciente(edad, peso, altura, sexo=0):
    """
    Convierte las características fisiológicas del paciente a los valores
    normalizados que espera el modelo predictor del BIS.

    Parámetros
    ----------
    edad, peso, altura : valores en unidades naturales (años, kg, cm)
    sexo               : 0 = masculino, 1 = femenino

    Retorna
    -------
    Tupla (edad_n, peso_n, altura_n, sexo_n)
    """
    return (
        (edad   - EDAD_MIN)   / (EDAD_MAX   - EDAD_MIN),
        (peso   - PESO_MIN)   / (PESO_MAX   - PESO_MIN),
        (altura - ALTURA_MIN) / (ALTURA_MAX - ALTURA_MIN),
        float(sexo),
    )


def hacer_cov_lote(df_pac: pd.DataFrame) -> torch.Tensor:
    """
    Construye un tensor de covariables (N, 4) a partir de un DataFrame de
    pacientes, lo que permite simular a toda una cohorte en un único lote,
    asignando a cada paciente sus propias características fisiológicas.

    Parámetros
    ----------
    df_pac : DataFrame con las columnas Edad, Peso, Altura y Sexo_num

    Retorna
    -------
    torch.Tensor de forma (N, 4) en el dispositivo activo
    """
    filas = [
        normalizar_paciente(float(f["Edad"]), float(f["Peso"]),
                             float(f["Altura"]), int(f["Sexo_num"]))
        for _, f in df_pac.iterrows()
    ]
    return torch.tensor(filas, dtype=torch.float32, device=dispositivo)


[INFO] Dispositivo : cuda
[INFO] GPU         : NVIDIA GeForce RTX 5060
[INFO] Modelo cargado — 144,212 parámetros


## Etapa 2: Filtro de señales, trayectoria Λ(t) y función de error ξ (tomada de `KNN_Estimacion_PID.ipynb`)

In [2]:
# =============================================================================
# FILTRO DE MEDIA EXPONENCIAL MÓVIL, TRAYECTORIA Y FUNCIÓN DE ERROR
# =============================================================================

# ── Hiperparámetros del filtro de media exponencial móvil ────────────────────
# Antes de introducirse al modelo predictor del BIS, las señales de control se
# filtran mediante una media exponencial móvil aplicada sobre una ventana
# deslizante de longitud fija, rellenada con ceros en su parte inicial:
#
#       y[n] = α · u[n] + (1 − α) · y[n − 1]
#
VENTANA_EMA = 35      # ← Longitud de la ventana deslizante
ALFA_EMA    = 0.01    # ← Factor de suavizado α


def construir_pesos_ema(ventana: int = VENTANA_EMA,
                         alfa: float = ALFA_EMA) -> torch.Tensor:
    """
    Construye el vector de pesos equivalente a la recursión de la media
    exponencial móvil sobre una ventana de longitud fija.

    Al reinicializarse la recursión con y[−1] = 0 al comienzo de cada ventana,
    la media exponencial móvil resulta ser una combinación lineal fija de las
    muestras contenidas en dicha ventana:

        y = Σ_{k=0}^{V−1}  α · (1 − α)^(V − 1 − k) · x[k]

    Expresarla de esta forma permite evaluarla mediante un único producto
    matricial, en lugar de un bucle recursivo, lo que reduce sustancialmente
    el tiempo de cómputo sin alterar en absoluto el resultado numérico.

    Parámetros
    ----------
    ventana : longitud de la ventana deslizante
    alfa    : factor de suavizado

    Retorna
    -------
    torch.Tensor de forma (ventana,) con los pesos del filtro
    """
    k = torch.arange(ventana, dtype=torch.float32, device=dispositivo)
    return alfa * (1.0 - alfa) ** (ventana - 1 - k)


@torch.no_grad()
def filtro_ema_ventana(buffer_crudo: torch.Tensor, t: int,
                        pesos_ema: torch.Tensor) -> torch.Tensor:
    """
    Calcula el valor filtrado en el instante t para todo el lote en paralelo.

    La ventana se conforma con las últimas V muestras de la señal cruda,
    completando con ceros las posiciones para las cuales aún no existe
    historial.

    Parámetros
    ----------
    buffer_crudo : (N, T_sim) — historial de la señal cruda de control
    t             : instante actual
    pesos_ema     : (V,) — pesos del filtro

    Retorna
    -------
    (N,) — valor filtrado en el instante t
    """
    N, V = buffer_crudo.shape[0], pesos_ema.shape[0]
    ini  = max(0, t - V + 1)
    fin  = t + 1
    n_relleno = V - (fin - ini)

    if n_relleno > 0:
        ceros = torch.zeros(N, n_relleno, device=buffer_crudo.device)
        ventana_datos = torch.cat([ceros, buffer_crudo[:, ini:fin]], dim=1)
    else:
        ventana_datos = buffer_crudo[:, ini:fin]

    return ventana_datos @ pesos_ema


@torch.no_grad()
def normalizar_lote(ventanas: torch.Tensor, n_pad: int) -> torch.Tensor:
    """
    Aplica normalización z-score por ventana a un lote de señales temporales,
    preservando en cero las n_pad primeras muestras de relleno.
    """
    T = ventanas.shape[1]
    if n_pad >= T:
        return torch.zeros_like(ventanas)
    porcion = ventanas[:, n_pad:]
    media   = porcion.mean(dim=1, keepdim=True)
    desvio  = porcion.std(dim=1, keepdim=True, correction=0).clamp(min=1e-8)
    out            = torch.zeros_like(ventanas)
    out[:, n_pad:] = (porcion - media) / desvio
    return out


# =============================================================================
# TRAYECTORIA DE REFERENCIA Λ(t)  (SEDACIÓN · MANTENIMIENTO · RECUPERACIÓN)
# =============================================================================
#   Fase 1 — Sedación      (   0 –  300 s): Λ(t) = 98 + 56.8·tanh(−0.004·t)
#   Fase 2 — Mantenimiento ( 300 – 3900 s): Λ(t) = 50  (constante)
#   Fase 3 — Recuperación  (3900 – 4440 s): Λ(t) = 50 + 49.7·tanh(0.004·(t−3900))

T_SEDACION      = 300     # [s]
T_MANTENIMIENTO = 3600    # [s]
T_RECUPERACION  = 540     # [s]

SED_A, SED_B, SED_K = 98.0, 56.8, -0.004
BIS_MANT            = 50.0
REC_A, REC_B, REC_K = 50.0, 49.7, 0.004

DT       = 10.0    # Período de muestreo [s]
T_VEN    = 180     # Longitud de ventana del modelo [muestras]
PPF_MAX  = 0.25    # Saturación máxima de propofol
REMI_MAX = 0.25    # Saturación máxima de remifentanilo

N_SED = int(T_SEDACION      / DT)
N_MAN = int(T_MANTENIMIENTO / DT)
N_REC = int(T_RECUPERACION  / DT)
N_TOT = N_SED + N_MAN + N_REC

LAMBDA_REF = np.zeros(N_TOT, dtype=np.float32)      # Λ(t)

for _i in range(N_SED):                                     # Fase 1
    LAMBDA_REF[_i] = SED_A + SED_B * np.tanh(SED_K * _i * DT)

LAMBDA_REF[N_SED : N_SED + N_MAN] = BIS_MANT                # Fase 2

for _i in range(N_REC):                                     # Fase 3
    LAMBDA_REF[N_SED + N_MAN + _i] = REC_A + REC_B * np.tanh(REC_K * _i * DT)

REF_TENSOR = torch.tensor(LAMBDA_REF, dtype=torch.float32, device=dispositivo)
EJE_T      = np.arange(N_TOT) * DT      # Eje temporal [s]
T_FINAL    = N_TOT * DT                 # Tf, duración total de la trayectoria
PESOS_EMA  = construir_pesos_ema()

print(f"[INFO] Trayectoria Λ(t): {N_TOT} muestras   Tf = {T_FINAL:.0f} s")
print(f"[INFO] Filtro EMA: ventana = {VENTANA_EMA}   α = {ALFA_EMA}   "
      f"(ganancia en régimen permanente = "
      f"{1 - (1 - ALFA_EMA) ** VENTANA_EMA:.4f})")


# =============================================================================
# SIMULACIÓN EN LOTE — PID PROPORCIONAL SOBRE LA TRAYECTORIA
# =============================================================================

@torch.no_grad()
def simular_pid_simple(poblacion, cov, *, ref_bis=None, bis_ini=None,
                        T_sim=N_TOT, T_ven=T_VEN, dt=DT,
                        ppf_max=PPF_MAX, remi_max=REMI_MAX,
                        pesos_ema=None):
    """
    Simula el lazo de control del PID proporcional, con filtrado por media
    exponencial móvil, para la trayectoria de referencia variable en el tiempo.
    

    Cada elemento del lote posee sus propias covariables, lo que permite
    simular a toda una cohorte de pacientes —cada uno con su propio conjunto
    de hiperparámetros— en una única llamada.

    Ley de control:
        e(t)         = BIS(t) − Λ(t)
        u_ppf_crudo  = clamp(Kp·e + Ki·∫e·dt + Kd·Δe/Δt, 0, ppf_max)
        u_remi_crudo = clamp(γ · u_ppf_crudo, 0, remi_max)

    Filtrado previo al ingreso a la planta:
        u_ppf  = EMA_ventana(u_ppf_crudo)
        u_remi = EMA_ventana(u_remi_crudo)

    Parámetros
    ----------
    poblacion : (N, 4) — [Kp, Ki, Kd, gamma] por paciente
    cov       : (N, 4) o (1, 4) — covariables

    Retorna
    -------
    ppf_filt_hist, remi_filt_hist, bis_hist — cada uno de forma (N, T_sim)
    """
    if ref_bis   is None: ref_bis   = REF_TENSOR
    if pesos_ema is None: pesos_ema = PESOS_EMA
    if bis_ini   is None: bis_ini   = float(ref_bis[0].item())

    N   = poblacion.shape[0]
    Kp, Ki, Kd, gam = (poblacion[:, 0], poblacion[:, 1],
                        poblacion[:, 2], poblacion[:, 3])
    cov_lote = cov if cov.shape[0] == N else cov.expand(N, -1)

    ppf_raw_h   = torch.zeros(N, T_sim,     dtype=torch.float32, device=dispositivo)
    remi_raw_h  = torch.zeros(N, T_sim,     dtype=torch.float32, device=dispositivo)
    ppf_filt_h  = torch.zeros(N, T_sim,     dtype=torch.float32, device=dispositivo)
    remi_filt_h = torch.zeros(N, T_sim,     dtype=torch.float32, device=dispositivo)
    bis_h       = torch.zeros(N, T_sim + 1, dtype=torch.float32, device=dispositivo)
    bis_h[:, 0] = bis_ini

    integr = torch.zeros(N, device=dispositivo)   # Integrador libre (sin anti-windup)
    e_prev = torch.zeros(N, device=dispositivo)

    for t in range(T_sim):
        e      = bis_h[:, t] - ref_bis[t]
        integr = integr + e * dt
        d_e    = (e - e_prev) / dt
        e_prev = e.clone()

        u_ppf_crudo  = (Kp * e + Ki * integr + Kd * d_e).clamp(0.0, ppf_max)
        u_remi_crudo = (gam * u_ppf_crudo).clamp(0.0, remi_max)
        ppf_raw_h[:, t]  = u_ppf_crudo
        remi_raw_h[:, t] = u_remi_crudo

        # Filtrado EMA antes de introducirse a la planta
        ppf_filt_h[:, t]  = filtro_ema_ventana(ppf_raw_h,  t, pesos_ema)
        remi_filt_h[:, t] = filtro_ema_ventana(remi_raw_h, t, pesos_ema)

        # Construcción de ventanas normalizadas a partir de la señal FILTRADA
        ini   = max(0, t - T_ven + 1)
        fin   = t + 1
        n_pad = T_ven - (fin - ini)
        if n_pad > 0:
            c   = torch.zeros(N, n_pad, device=dispositivo)
            v_p = torch.cat([c, ppf_filt_h[:, ini:fin]],  dim=1)
            v_r = torch.cat([c, remi_filt_h[:, ini:fin]], dim=1)
            v_b = torch.cat([c, bis_h[:, ini:fin]],       dim=1)
        else:
            v_p, v_r, v_b = (ppf_filt_h[:, ini:fin],
                              remi_filt_h[:, ini:fin],
                              bis_h[:, ini:fin])

        inf_t = torch.stack([normalizar_lote(v_p, n_pad),
                              normalizar_lote(v_r, n_pad)], dim=1)
        bis_pred, _ = modelo(inf_t, normalizar_lote(v_b, n_pad), cov_lote)
        bis_h[:, t + 1] = bis_pred

    return ppf_filt_h, remi_filt_h, bis_h[:, 1:]


# =============================================================================
# FUNCIÓN DE ERROR ξ 
# =============================================================================

def calcular_xi(bis_np: np.ndarray) -> np.ndarray:
    """
    Evalúa la función de error ξ, definida como la integral del tiempo
    multiplicado por el valor absoluto del error relativo a la trayectoria de
    referencia, normalizada por el cuadrado de la duración total:

              2      ⌠ Tf        | e(t) |
        ξ  = ────  · ⎮      t · │────────│ dt
             Tf²     ⌡ 0        |  Λ(t)  |

    donde e(t) = BIS(t) − Λ(t) y Λ(t) es la trayectoria de sedación,
    mantenimiento y recuperación.

    El factor de normalización 2/Tf² confiere a ξ un carácter adimensional:
    si el error relativo permaneciera constante e igual a c a lo largo de toda
    la simulación, se obtendría exactamente ξ = c. En consecuencia, ξ admite
    una interpretación directa como error relativo medio ponderado en el
    tiempo, penalizando con mayor severidad las desviaciones tardías.

    La integral se aproxima numéricamente mediante suma de Riemann con paso DT.

    Parámetros
    ----------
    bis_np : (N, T) — respuesta BIS simulada de cada paciente

    Retorna
    -------
    (N,) — valor de ξ para cada paciente
    """
    error_rel = np.abs((bis_np - LAMBDA_REF[None, :]) / LAMBDA_REF[None, :])
    integral  = np.sum(EJE_T[None, :] * error_rel, axis=1) * DT
    return (2.0 / T_FINAL ** 2) * integral


def evaluar_cohorte(params_np: np.ndarray, df_pac: pd.DataFrame) -> tuple:
    """
    Simula la trayectoria completa para una cohorte de pacientes, cada uno con
    su propio conjunto de hiperparámetros, y evalúa la función de error ξ.

    Parámetros
    ----------
    params_np : (N, 4) — hiperparámetros [Kp, Ki, Kd, gamma] por paciente
    df_pac    : DataFrame de N pacientes

    Retorna
    -------
    xi     : (N,)       — valor de la función de error de cada paciente
    bis_np : (N, N_TOT) — respuesta BIS simulada
    """
    params_t = torch.tensor(params_np, dtype=torch.float32, device=dispositivo)
    cov_t    = hacer_cov_lote(df_pac)
    _, _, bis_t = simular_pid_simple(params_t, cov_t)
    bis_np = bis_t.cpu().numpy()
    return calcular_xi(bis_np), bis_np


[INFO] Trayectoria Λ(t): 444 muestras   Tf = 4440 s
[INFO] Filtro EMA: ventana = 35   α = 0.01   (ganancia en régimen permanente = 0.2966)


## Etapa 3: Conformación de los conjuntos (tomada de `KNN_Estimacion_PID.ipynb`)

In [3]:
# =============================================================================
# CARGA DE DATOS Y CONFORMACIÓN DE LOS CONJUNTOS
# =============================================================================
#
#   1. Datos de VALIDACIÓN : registros cuyo RMSE es estrictamente mayor a 16.
#      Se descartan del proceso de estimación y se reservan para validación.
#
#   2. Datos de ESTIMACIÓN : registros cuyo RMSE es menor o igual a 16. Este
#      conjunto se subdivide en:
#
#         · ESPACIO (Universo) — conjunto de referencia del algoritmo de K
#           vecinos más cercanos. Se conforma con los pacientes de MENOR error,
#           ordenados de forma ascendente por RMSE, e imponiendo una
#           composición equilibrada del 50 % de hombres y 50 % de mujeres. Es
#           decir, se seleccionan los hombres con menor error y las mujeres con
#           menor error, en igual número.
#
#         · DATOS DE ESTIMACIÓN — los pacientes restantes, cuyos
#           hiperparámetros serán estimados y evaluados.
#
#   Los espacios de los algoritmos separados por sexo se obtienen filtrando el
#   espacio global, por lo que quedan conformados, respectivamente, por los
#   hombres con menor error y por las mujeres con menor error. De esta manera
#   los tres algoritmos operan sobre particiones consistentes entre sí.

UMBRAL_RMSE   = 16.0   # ← Umbral de descarte
FRAC_ESPACIO  = 0.40   # ← Proporción de los datos destinada al Espacio

# ── Lectura del archivo ───────────────────────────────────────────────────────
print(f"[INFO] Cargando: {RUTA_SINTONIZACIONES}")
try:
    df_crudo = pd.read_csv(RUTA_SINTONIZACIONES)
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo '{RUTA_SINTONIZACIONES}'. Verifique que "
        f"se encuentre en la misma carpeta del proyecto."
    )

columnas_requeridas = {"Edad", "Peso", "Altura", "Sexo",
                       "Kp", "Ki", "Kd", "gamma", "RMSE"}
faltantes = columnas_requeridas - set(df_crudo.columns)
if faltantes:
    raise ValueError(f"El archivo no contiene las columnas esperadas: {faltantes}. "
                     f"Columnas encontradas: {list(df_crudo.columns)}")

# Codificación numérica del sexo: 'M' → 0, 'F' → 1
df_crudo["Sexo_num"] = (
    df_crudo["Sexo"].astype(str).str.strip().str.upper().map({"M": 0, "F": 1})
)
if df_crudo["Sexo_num"].isna().any():
    raise ValueError("Se encontraron valores de Sexo no reconocidos "
                     "(solo se aceptan 'M' y 'F').")

print(f"[INFO] Registros totales: {len(df_crudo)}")

# ── Partición 1: validación (RMSE > 16) frente a estimación (RMSE ≤ 16) ──────
df_validacion = (
    df_crudo[df_crudo["RMSE"] > UMBRAL_RMSE].copy().reset_index(drop=True)
)
df_estimacion_total = (
    df_crudo[df_crudo["RMSE"] <= UMBRAL_RMSE].copy().reset_index(drop=True)
)

print(f"[INFO] Datos de VALIDACIÓN (RMSE > {UMBRAL_RMSE}) : "
      f"{len(df_validacion)} pacientes")
print(f"[INFO] Datos de ESTIMACIÓN (RMSE ≤ {UMBRAL_RMSE}) : "
      f"{len(df_estimacion_total)} pacientes")

# ── Partición 2: Espacio equilibrado por sexo, seleccionado por menor RMSE ───
df_h_orden = (df_estimacion_total[df_estimacion_total["Sexo_num"] == 0]
              .sort_values("RMSE", ascending=True))
df_m_orden = (df_estimacion_total[df_estimacion_total["Sexo_num"] == 1]
              .sort_values("RMSE", ascending=True))

n_espacio_objetivo = int(round(FRAC_ESPACIO * len(df_estimacion_total)))
# Se toma el mismo número de hombres y de mujeres, garantizando la
# composición equilibrada del 50 % solicitada.
n_por_sexo = max(1, min(n_espacio_objetivo // 2,
                         len(df_h_orden) - 1, len(df_m_orden) - 1))

idx_espacio = list(df_h_orden.index[:n_por_sexo]) + \
              list(df_m_orden.index[:n_por_sexo])

df_espacio = (df_estimacion_total.loc[idx_espacio]
              .copy().reset_index(drop=True))
df_estimados = (df_estimacion_total.drop(index=idx_espacio)
                .copy().reset_index(drop=True))

print(f"\n[INFO] ESPACIO (referencia)  : {len(df_espacio)} pacientes  "
      f"→ {n_por_sexo} hombres + {n_por_sexo} mujeres (50 % / 50 %)")
print(f"[INFO] DATOS DE ESTIMACIÓN   : {len(df_estimados)} pacientes  "
      f"→ hombres: {(df_estimados['Sexo_num']==0).sum()}   "
      f"mujeres: {(df_estimados['Sexo_num']==1).sum()}")
print(f"       RMSE del Espacio      : mín {df_espacio['RMSE'].min():.4f}   "
      f"máx {df_espacio['RMSE'].max():.4f}")

if len(df_estimados) == 0:
    raise ValueError("No quedaron pacientes en el conjunto de estimación. "
                     "Reduzca el valor de FRAC_ESPACIO.")

# ── Subconjuntos por sexo, derivados de la partición global ──────────────────
df_espacio_h   = df_espacio[df_espacio["Sexo_num"] == 0].reset_index(drop=True)
df_espacio_m   = df_espacio[df_espacio["Sexo_num"] == 1].reset_index(drop=True)
df_estimados_h = df_estimados[df_estimados["Sexo_num"] == 0].reset_index(drop=True)
df_estimados_m = df_estimados[df_estimados["Sexo_num"] == 1].reset_index(drop=True)

print(f"\n[INFO] Subconjunto masculino : espacio {len(df_espacio_h)}  |  "
      f"a estimar {len(df_estimados_h)}")
print(f"[INFO] Subconjunto femenino  : espacio {len(df_espacio_m)}  |  "
      f"a estimar {len(df_estimados_m)}")

# ── Vista previa ──────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  ESPACIO — pacientes de menor RMSE, equilibrado por sexo")
print("=" * 72)
try:
    display(df_espacio.sort_values("RMSE"))
except NameError:
    print(df_espacio.sort_values("RMSE").to_string(index=False))

print("\n" + "=" * 72)
print("  DATOS DE ESTIMACIÓN")
print("=" * 72)
try:
    display(df_estimados.head(12))
except NameError:
    print(df_estimados.head(12).to_string(index=False))


[INFO] Cargando: SINTONIZACIONES.csv
[INFO] Registros totales: 340
[INFO] Datos de VALIDACIÓN (RMSE > 16.0) : 50 pacientes
[INFO] Datos de ESTIMACIÓN (RMSE ≤ 16.0) : 290 pacientes

[INFO] ESPACIO (referencia)  : 116 pacientes  → 58 hombres + 58 mujeres (50 % / 50 %)
[INFO] DATOS DE ESTIMACIÓN   : 174 pacientes  → hombres: 76   mujeres: 98
       RMSE del Espacio      : mín 4.4912   máx 9.2380

[INFO] Subconjunto masculino : espacio 58  |  a estimar 76
[INFO] Subconjunto femenino  : espacio 58  |  a estimar 98

  ESPACIO — pacientes de menor RMSE, equilibrado por sexo


,Edad,Peso,Altura,Sexo,Kp,Ki,Kd,gamma,RMSE,Sexo_num
58,56.0,48.50,152.0,F,2.473780,0.007955,2.842341,0.793992,4.491245,1
59,73.0,43.40,151.8,F,2.556277,0.005023,1.968817,2.466328,4.982819,1
60,46.0,53.00,155.4,F,2.328256,0.001036,2.516666,1.727226,4.984394,1
61,53.0,54.50,151.9,F,2.412171,0.002763,2.174127,1.178737,5.177191,1
62,50.0,48.80,152.0,F,3.000000,0.002331,1.019977,0.933848,5.208674,1
...,...,...,...,...,...,...,...,...,...,...
53,63.0,64.70,169.7,M,1.447385,0.069082,1.804527,2.419233,9.174468,0
54,69.0,51.20,170.0,M,2.911389,0.010683,2.465602,2.096380,9.185011,0
55,63.0,59.00,158.0,M,1.573212,0.002915,2.193878,2.355651,9.221971,0
56,45.0,74.05,171.6,M,2.081540,0.008805,2.104569,1.567199,9.226124,0



  DATOS DE ESTIMACIÓN


,Edad,Peso,Altura,Sexo,Kp,Ki,Kd,gamma,RMSE,Sexo_num
0,46.0,81.80,174.5,M,1.156239,0.001394,0.477922,1.865874,12.129439,0
1,38.0,62.00,175.0,M,1.841619,0.011363,0.380515,0.025481,15.867215,0
2,58.0,82.80,166.4,M,1.812955,0.002014,1.748188,2.533071,14.627651,0
3,58.0,70.45,182.0,M,2.418994,0.005543,1.115765,1.052635,9.401891,0
4,55.0,51.50,168.8,M,1.823172,0.009494,0.060461,0.973162,14.747317,0
5,76.0,92.00,164.5,M,2.228152,0.001597,2.543940,0.191922,9.426901,0
6,81.0,59.60,162.9,M,3.000000,0.012905,1.446171,2.991271,9.260944,0
7,47.0,61.80,172.0,M,1.114274,0.002353,2.549404,1.087227,9.619498,0
8,82.0,51.80,154.2,M,1.607137,0.004001,0.430306,2.164635,15.524870,0
9,60.0,78.30,171.7,M,2.241705,0.007498,1.373938,2.323957,9.499188,0


## Etapa 4: Funciones del algoritmo de K vecinos más cercanos (tomada de `KNN_Estimacion_PID.ipynb`)

In [4]:
# =============================================================================
# ALGORITMO DE K VECINOS MÁS CERCANOS — FUNCIONES BASE
# =============================================================================
#
# Procedimiento de estimación de los hiperparámetros del PID proporcional:
#
#   1. Los vectores de características del Espacio se normalizan mediante
#      min-max al intervalo [0, 1]. La variable Sexo, por ser categórica
#      binaria, se codifica directamente como 0 y 1 sin reescalamiento.
#
#   2. Cada paciente a estimar se normaliza con esos mismos rangos y se
#      calculan sus distancias de Minkowski respecto a todo el Espacio.
#
#   3. Se seleccionan los K vecinos más cercanos, se toman los inversos de sus
#      distancias (1/dᵢ) y se transforman mediante la función softmax, de modo
#      que los pesos resultantes sumen la unidad.
#
#   4. La estimación es la combinación lineal de los vectores de
#      hiperparámetros de dichos vecinos, ponderada por esos pesos, y se acota
#      finalmente al intervalo admisible.
#
#   5. El desempeño de cada valor del exponente p se evalúa simulando el lazo
#      de control y calculando la función de error ξ.

COTA_INF, COTA_SUP = 1e-8, 3.0             # ← Cotas de la estimación final
COLS_HIPER = ["Kp", "Ki", "Kd", "gamma"]   # Hiperparámetros a estimar


def generar_valores_p(p_inicio: float, p_fin: float,
                       p_incremento: float) -> list:
    """
    Genera la lista de exponentes p de la distancia de Minkowski a evaluar,
    a partir del valor inicial, el valor final y el incremento especificados.

    Parámetros
    ----------
    p_inicio     : primer valor del exponente
    p_fin        : último valor del exponente (inclusive)
    p_incremento : paso entre valores consecutivos

    Retorna
    -------
    list de floats con los valores del exponente
    """
    n_pasos = int(round((p_fin - p_inicio) / p_incremento)) + 1
    return [round(p_inicio + i * p_incremento, 10) for i in range(n_pasos)]


def ajustar_normalizador(df_ref: pd.DataFrame, columnas: list) -> dict:
    """
    Calcula los rangos min-max de las variables a partir del Espacio.
    """
    return {c: (float(df_ref[c].min()), float(df_ref[c].max()))
            for c in columnas}


def aplicar_normalizador(df_dat: pd.DataFrame, columnas: list,
                          rangos: dict) -> np.ndarray:
    """
    Construye la matriz de vectores de características normalizados.

    Las variables continuas se reescalan al intervalo [0, 1] con los rangos
    del Espacio. La variable Sexo se emplea directamente con sus valores 0 y 1.
    Si el rango de alguna variable resulta nulo, dicha variable se fija en cero
    para evitar una división entre cero.
    """
    X = np.zeros((len(df_dat), len(columnas)), dtype=np.float64)
    for j, c in enumerate(columnas):
        valores = df_dat[c].to_numpy(dtype=np.float64)
        if c == "Sexo_num":
            X[:, j] = valores          # Categórica binaria: sin reescalamiento
        else:
            v_min, v_max = rangos[c]
            X[:, j] = 0.0 if v_max == v_min else (valores - v_min) / (v_max - v_min)
    return X


def distancia_minkowski(X_ref: np.ndarray, x_consulta: np.ndarray,
                         p: float) -> np.ndarray:
    """
    Distancia de Minkowski de orden p:  d(x,y) = ( Σ |xᵢ − yᵢ|^p )^(1/p)
    """
    return np.sum(np.abs(X_ref - x_consulta[None, :]) ** p, axis=1) ** (1.0 / p)


def pesos_softmax_inversos(distancias: np.ndarray) -> np.ndarray:
    """
    Calcula los pesos de la combinación lineal a partir de los inversos de las
    distancias de los K vecinos, transformados mediante la función softmax:

        wᵢ = exp(1/dᵢ) / Σⱼ exp(1/dⱼ)

    Se sustrae el máximo antes de exponenciar, procedimiento numéricamente
    estable que evita desbordamientos cuando alguna distancia es muy próxima
    a cero.
    """
    inversos   = 1.0 / np.maximum(distancias, 1e-12)
    exponentes = np.exp(inversos - np.max(inversos))
    return exponentes / np.sum(exponentes)


def estimar_hiperparametros_knn(X_espacio: np.ndarray, Y_espacio: np.ndarray,
                                 X_consulta: np.ndarray, k: int, p: float,
                                 cota_inf: float = COTA_INF,
                                 cota_sup: float = COTA_SUP) -> np.ndarray:
    """
    Estima los hiperparámetros del controlador mediante K vecinos más cercanos
    con ponderación softmax de los inversos de las distancias de Minkowski.

    Parámetros
    ----------
    X_espacio  : (M, D) — vectores normalizados del Espacio
    Y_espacio  : (M, H) — hiperparámetros asociados a cada elemento del Espacio
    X_consulta : (N, D) — vectores normalizados de los pacientes a estimar
    k          : número de vecinos
    p          : orden de la distancia de Minkowski

    Retorna
    -------
    (N, H) — hiperparámetros estimados, acotados al intervalo admisible
    """
    N, H = X_consulta.shape[0], Y_espacio.shape[1]
    Y_est = np.zeros((N, H), dtype=np.float64)
    k_efectivo = min(k, X_espacio.shape[0])

    for i in range(N):
        d = distancia_minkowski(X_espacio, X_consulta[i], p)
        idx_vecinos = np.argsort(d)[:k_efectivo]
        w = pesos_softmax_inversos(d[idx_vecinos])
        Y_est[i] = w @ Y_espacio[idx_vecinos]

    return np.clip(Y_est, cota_inf, cota_sup)


def barrido_exponente_p(X_espacio, Y_espacio, X_consulta, Y_real,
                         df_consulta, k, valores_p, mostrar_progreso=True):
    """
    Ejecuta la estimación para cada valor del exponente p, simula el lazo de
    control resultante y evalúa la función de error ξ.

    Adicionalmente se registra, con carácter informativo, el error absoluto
    medio de cada hiperparámetro respecto a los valores obtenidos previamente
    mediante Algoritmo Genético.

    Retorna
    -------
    tabla     : DataFrame con una fila por valor de p
    registros : dict {p: (Y_est, xi, bis)} con los resultados detallados
    """
    filas, registros = [], {}
    t0 = time.time()

    for n_p, p in enumerate(valores_p, start=1):
        Y_est = estimar_hiperparametros_knn(X_espacio, Y_espacio,
                                             X_consulta, k, p)
        xi, bis = evaluar_cohorte(Y_est, df_consulta)

        fila = {"p": p,
                "xi medio": float(np.mean(xi)),
                "xi desv.est.": float(np.std(xi, ddof=1)) if len(xi) > 1 else np.nan,
                "xi mínimo": float(np.min(xi))}
        # Error absoluto medio por hiperparámetro (informativo)
        for j, c in enumerate(COLS_HIPER):
            fila[f"EAM {c}"] = float(np.mean(np.abs(Y_est[:, j] - Y_real[:, j])))

        filas.append(fila)
        registros[p] = (Y_est, xi, bis)

        if mostrar_progreso:
            transcurrido = time.time() - t0
            eta = transcurrido * (len(valores_p) - n_p) / n_p
            sys.stdout.write(
                f"\r    p = {p:<6.3f}  ({n_p:>3d}/{len(valores_p)})  "
                f"ξ medio = {fila['xi medio']:.6f}   "
                f"[{transcurrido:5.0f}s, ETA {eta:5.0f}s]        ")
            sys.stdout.flush()

    if mostrar_progreso:
        sys.stdout.write("\n")

    return pd.DataFrame(filas), registros


def ejecutar_algoritmo_knn(df_espacio_alg, df_estimados_alg, columnas_vector,
                            k, valores_p, nombre_algoritmo) -> dict:
    """
    Ejecuta el procedimiento completo de un algoritmo: normalización, barrido
    sobre el exponente p, selección del valor con menor ξ medio y presentación
    de los resultados.

    Parámetros
    ----------
    df_espacio_alg   : DataFrame del Espacio del algoritmo
    df_estimados_alg : DataFrame de los pacientes a estimar
    columnas_vector  : variables que conforman el vector de características
    k                : número de vecinos del algoritmo
    valores_p        : lista de exponentes de Minkowski a evaluar
    nombre_algoritmo : denominación del algoritmo

    Retorna
    -------
    dict con: nombre, tabla_p, mejor_p, k, Y_est, xi, bis, df
    """
    print("\n" + "=" * 78)
    print(f"  {nombre_algoritmo}")
    print(f"  Vector de características : {columnas_vector}")
    print(f"  Espacio: {len(df_espacio_alg)} pacientes   |   "
          f"A estimar: {len(df_estimados_alg)} pacientes   |   K = {k}")
    print(f"  Exponentes p evaluados    : {len(valores_p)}  "
          f"(de {valores_p[0]} a {valores_p[-1]})")
    print("=" * 78)

    if len(df_espacio_alg) < k:
        raise ValueError(f"El Espacio ({len(df_espacio_alg)}) es menor que K = {k}.")
    if len(df_estimados_alg) == 0:
        raise ValueError("No hay pacientes que estimar en este algoritmo.")

    rangos = ajustar_normalizador(df_espacio_alg, columnas_vector)
    X_esp  = aplicar_normalizador(df_espacio_alg,  columnas_vector, rangos)
    X_est  = aplicar_normalizador(df_estimados_alg, columnas_vector, rangos)
    Y_esp  = df_espacio_alg[COLS_HIPER].to_numpy(dtype=np.float64)
    Y_real = df_estimados_alg[COLS_HIPER].to_numpy(dtype=np.float64)

    print("\n  Barrido sobre el exponente de la distancia de Minkowski:")
    tabla_p, registros = barrido_exponente_p(
        X_esp, Y_esp, X_est, Y_real, df_estimados_alg, k, valores_p
    )

    print("\n  RESULTADOS POR VALOR DE p")
    print("  " + "─" * 74)
    try:
        display(tabla_p)
    except NameError:
        print(tabla_p.to_string(index=False))

    idx_mejor = int(tabla_p["xi medio"].idxmin())
    mejor_p   = float(tabla_p.loc[idx_mejor, "p"])
    Y_est, xi, bis = registros[mejor_p]

    print(f"\n  ► Mejor exponente: p = {mejor_p}   "
          f"(ξ medio = {tabla_p.loc[idx_mejor, 'xi medio']:.6f})")
    print("  " + "─" * 74)
    print(f"  {'ξ medio':<24}: {np.mean(xi):.6f}")
    print(f"  {'ξ desviación estándar':<24}: "
          f"{np.std(xi, ddof=1) if len(xi) > 1 else float('nan'):.6f}")
    print(f"  {'ξ mínimo / máximo':<24}: {np.min(xi):.6f} / {np.max(xi):.6f}")
    print("=" * 78)

    return {"nombre": nombre_algoritmo, "tabla_p": tabla_p, "mejor_p": mejor_p,
            "k": k, "Y_est": Y_est, "xi": xi, "bis": bis,
            "df": df_estimados_alg.reset_index(drop=True)}


## Etapa 5: Estimación de parámetros y tiempo de cómputo

In [5]:
# =============================================================================
# ESTIMACIÓN DE LOS PARÁMETROS CON LAS TRES ESTRATEGIAS
# =============================================================================
K_KNN, P_KNN = 2, 1.4                                 # Variante global seleccionada
COLUMNAS_KNN = ["Edad", "Peso", "Altura", "Sexo_num"]

rangos_knn = ajustar_normalizador(df_espacio, COLUMNAS_KNN)
X_espacio  = aplicar_normalizador(df_espacio, COLUMNAS_KNN, rangos_knn)
Y_espacio  = df_espacio[COLS_HIPER].to_numpy(dtype=np.float64)

# Tiempo de cómputo de la estimación para el conjunto de estimación
X_estimados = aplicar_normalizador(df_estimados, COLUMNAS_KNN, rangos_knn)
t0 = time.perf_counter()
Y_knn_estimados = estimar_hiperparametros_knn(X_espacio, Y_espacio, X_estimados, K_KNN, P_KNN)
t_knn = time.perf_counter() - t0
print(f"[INFO] Estimación KNN de {len(X_estimados)} pacientes: {t_knn * 1e3:.2f} ms "
      f"({t_knn / len(X_estimados) * 1e6:.1f} µs por paciente)")

X_deficientes   = aplicar_normalizador(df_validacion, COLUMNAS_KNN, rangos_knn)
Y_knn_deficientes = estimar_hiperparametros_knn(X_espacio, Y_espacio, X_deficientes, K_KNN, P_KNN)

# Controlador de parámetros fijos: mediana del espacio de referencia
mediana_espacio = np.median(Y_espacio, axis=0)
print("[INFO] Parámetros medianos [Kp, Ki, Kd, gamma]:", np.round(mediana_espacio, 5))

estrategias = {
    ("Estimación", "AG"):      (df_estimados[COLS_HIPER].to_numpy(dtype=np.float64), df_estimados),
    ("Estimación", "KNN"):     (Y_knn_estimados, df_estimados),
    ("Estimación", "Mediana"): (np.tile(mediana_espacio, (len(df_estimados), 1)), df_estimados),
    ("Deficientes", "AG"):     (df_validacion[COLS_HIPER].to_numpy(dtype=np.float64), df_validacion),
    ("Deficientes", "KNN"):    (Y_knn_deficientes, df_validacion),
    ("Deficientes", "Mediana"):(np.tile(mediana_espacio, (len(df_validacion), 1)), df_validacion),
}

[INFO] Estimación KNN de 174 pacientes: 4.13 ms (23.7 µs por paciente)
[INFO] Parámetros medianos [Kp, Ki, Kd, gamma]: [1.95493 0.00209 1.58535 1.57095]


## Etapa 6: Simulación en lazo cerrado y métricas de desempeño

In [6]:
# =============================================================================
# SIMULACIÓN DE LAS TRES ESTRATEGIAS Y CÁLCULO DE MÉTRICAS
# =============================================================================
def metricas_fase(bis_np):
    """
    Calcula, para cada paciente, el RMSE respecto de la trayectoria de
    referencia, el porcentaje del tiempo de mantenimiento con el BIS entre 40 y
    60, el RMSE durante el mantenimiento y el BIS al final de la recuperación.
    """
    rmse  = np.sqrt(np.mean((bis_np - LAMBDA_REF[None, :]) ** 2, axis=1))
    mant  = bis_np[:, N_SED:N_SED + N_MAN]
    pct   = np.mean((mant >= 40) & (mant <= 60), axis=1) * 100
    rmse_mant = np.sqrt(np.mean((mant - BIS_MANT) ** 2, axis=1))
    return rmse, pct, rmse_mant, bis_np[:, -1]


filas, trayectorias = [], {}
for (conjunto, estrategia), (parametros, df_pac) in estrategias.items():
    t0 = time.time()
    xi, bis_np = evaluar_cohorte(parametros, df_pac)
    rmse, pct, rmse_mant, bis_final = metricas_fase(bis_np)
    trayectorias[f"{conjunto}_{estrategia}"] = bis_np
    for i in range(len(df_pac)):
        filas.append(dict(Conjunto=conjunto, Estrategia=estrategia,
                          Edad=df_pac["Edad"].iloc[i], Peso=df_pac["Peso"].iloc[i],
                          Altura=df_pac["Altura"].iloc[i], Sexo=df_pac["Sexo"].iloc[i],
                          xi=xi[i], RMSE=rmse[i], Pct_40_60_mantenimiento=pct[i],
                          RMSE_mantenimiento=rmse_mant[i], BIS_final=bis_final[i]))
    print(f"  {conjunto:12s} {estrategia:8s}  xi = {xi.mean():.4f} ± {xi.std(ddof=1):.4f}   "
          f"RMSE = {rmse.mean():.2f}   [{time.time() - t0:.0f} s]")

resultados = pd.DataFrame(filas)
resultados.to_csv("Resultados_Comparacion_KNN_AG_Mediana.csv", index=False)
np.savez_compressed("Trayectorias_BIS_Comparacion.npz", **trayectorias)
print("[INFO] Resultados guardados en Resultados_Comparacion_KNN_AG_Mediana.csv "
      "y Trayectorias_BIS_Comparacion.npz")

  Estimación   AG        xi = 0.1114 ± 0.0381   RMSE = 9.31   [3 s]
  Estimación   KNN       xi = 0.1626 ± 0.0636   RMSE = 12.96   [3 s]
  Estimación   Mediana   xi = 0.1734 ± 0.0765   RMSE = 13.84   [3 s]
  Deficientes  AG        xi = 0.1363 ± 0.0377   RMSE = 10.95   [2 s]
  Deficientes  KNN       xi = 0.1763 ± 0.0485   RMSE = 13.88   [2 s]
  Deficientes  Mediana   xi = 0.1746 ± 0.0479   RMSE = 14.14   [2 s]
[INFO] Resultados guardados en Resultados_Comparacion_KNN_AG_Mediana.csv y Trayectorias_BIS_Comparacion.npz


## Etapa 7: Tabla resumen y pruebas estadísticas

In [7]:
# =============================================================================
# TABLA RESUMEN (MEDIA ± DESVIACIÓN ESTÁNDAR)
# =============================================================================
METRICAS_COMP = ["xi", "RMSE", "Pct_40_60_mantenimiento", "BIS_final"]
resumen = (resultados.groupby(["Conjunto", "Estrategia"])[METRICAS_COMP]
                     .agg(["mean", "std"]).round(3))
resumen

xi           RMSE        Pct_40_60_mantenimiento  \
                         mean    std    mean    std                    mean   
Conjunto    Estrategia                                                        
Deficientes AG          0.136  0.038  10.947  3.378                  89.933   
            KNN         0.176  0.048  13.884  3.080                  86.167   
            Mediana     0.175  0.048  14.144  2.911                  87.861   
Estimación  AG          0.111  0.038   9.310  3.129                  93.683   
            KNN         0.163  0.064  12.957  3.717                  87.929   
            Mediana     0.173  0.077  13.840  4.013                  87.936   

                                BIS_final          
                           std       mean     std  
Conjunto    Estrategia                             
Deficientes AG           6.585  64.457001  13.090  
            KNN          8.796  62.313000  12.474  
            Mediana     10.342  61.812000  10.282  
Estimación  AG           6.140  69.362999  10.070  
            KNN         12.462  65.365997  12.264  
            Mediana     13.358  64.422997  11.534

In [8]:
# =============================================================================
# COMPARACIONES APAREADAS MEDIANTE LA PRUEBA DE RANGOS CON SIGNO DE WILCOXON
# =============================================================================
COMPARACIONES = {"Estimación": [("KNN", "AG"), ("KNN", "Mediana"), ("AG", "Mediana")],
                 "Deficientes": [("KNN", "AG"), ("KNN", "Mediana")]}
filas = []
for conjunto, pares in COMPARACIONES.items():
    sub = resultados[resultados.Conjunto == conjunto]
    for a, b in pares:
        fila = dict(Conjunto=conjunto, Comparacion=f"{a} frente a {b}")
        for metrica in ["xi", "RMSE"]:
            va = sub.loc[sub.Estrategia == a, metrica].to_numpy()
            vb = sub.loc[sub.Estrategia == b, metrica].to_numpy()
            fila[f"Dif_mediana_{metrica}"] = float(np.median(va - vb))
            fila[f"Valor_p_{metrica}"] = stats.wilcoxon(va, vb).pvalue
            fila[f"Primero_mejor_{metrica}"] = f"{int((va < vb).sum())}/{len(va)}"
        filas.append(fila)
pruebas = pd.DataFrame(filas)
pruebas

,Conjunto,Comparacion,Dif_mediana_xi,Valor_p_xi,Primero_mejor_xi,Dif_mediana_RMSE,Valor_p_RMSE,Primero_mejor_RMSE
0,Estimación,KNN frente a AG,0.043511,5.205169e-19,30/174,3.583851,6.724511e-17,34/174
1,Estimación,KNN frente a Mediana,-0.006112,4.534208e-02,105/174,-0.427471,1.388719e-02,103/174
2,Estimación,AG frente a Mediana,-0.053787,7.629796e-21,146/174,-4.617789,1.861801e-20,147/174
3,Deficientes,KNN frente a AG,0.041214,1.045182e-05,12/50,3.351075,6.508991e-05,12/50
4,Deficientes,KNN frente a Mediana,-0.001579,9.326176e-01,26/50,-0.425044,6.724704e-01,29/50


In [9]:
# =============================================================================
# RAZÓN DE DESEMPEÑO ENTRE LA HEURÍSTICA Y LA SINTONIZACIÓN INDIVIDUAL
# =============================================================================
sub = resultados[resultados.Conjunto == "Estimación"]
xi_knn = sub.loc[sub.Estrategia == "KNN", "xi"].to_numpy()
xi_ag  = sub.loc[sub.Estrategia == "AG", "xi"].to_numpy()
xi_med = sub.loc[sub.Estrategia == "Mediana", "xi"].to_numpy()
print(f"Razón mediana xi_KNN / xi_AG por paciente: {np.median(xi_knn / xi_ag):.3f}")
recuperado = (xi_med.mean() - xi_knn.mean()) / (xi_med.mean() - xi_ag.mean()) * 100
print(f"Fracción de la mejora en xi recuperada por la heurística: {recuperado:.0f} %")

Razón mediana xi_KNN / xi_AG por paciente: 1.412
Fracción de la mejora en xi recuperada por la heurística: 17 %
